# Block C: Segmentation Strategy (Overlapping Windows)

Goal: Convert continuous CTU-CHB recordings into fixed-length overlapping windows suitable for:
- Random Forest (features computed per window in Block D)
- CNN (raw window inputs in Block G)
- Real-time risk timeline (Block F)

Chosen streaming configuration:
- Window length: 10 minutes
- Step (stride): 1 minute
- Sampling rate: 4 Hz  → window = 2400 samples, step = 240 samples

This notebook:
- Loads Block A metadata (labels + record list)
- Loads each recording, preprocesses FHR/UC (Block B logic)
- Creates overlapping windows and computes per-window QC
- Saves a window manifest CSV: `outputs/segmentation_manifest.csv`


In [15]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

import wfdb
from scipy.ndimage import gaussian_filter1d


In [16]:
data_dir = Path("../data/ctu-chb-intrapartum-cardiotocography-database-1.0.0").resolve()
out_dir = Path("../outputs").resolve()
out_dir.mkdir(parents=True, exist_ok=True)

summary_csv = out_dir / "dataset_summary.csv"
stats_json = out_dir / "dataset_statistics.json"

dfA = pd.read_csv(summary_csv)
with open(stats_json, "r") as f:
    statsA = json.load(f)

print("Loaded Block A summary:", dfA.shape)
print("Unknown record IDs:", statsA.get("unknown_record_ids", []))
print("Failed-quality record IDs (n):", len(statsA.get("failed_quality_record_ids", [])))


Loaded Block A summary: (552, 22)
Unknown record IDs: ['1002', '1017']
Failed-quality record IDs (n): 10


In [17]:
FS = 4.0  # CTU-CHB is 4 Hz

WINDOW_MIN = 10
STEP_MIN = 1

WINDOW_SAMPLES = int(WINDOW_MIN * 60 * FS)  # 2400
STEP_SAMPLES   = int(STEP_MIN * 60 * FS)    # 240

# Window QC thresholds (post-preprocess)
MIN_VALID_PCT_POST = 70.0           # keep if >= 70% valid after preprocess
MAX_GAP_SEC_POST = 120.0            # keep if longest missing gap <= 120s (optional but helpful)

print("WINDOW_SAMPLES:", WINDOW_SAMPLES)
print("STEP_SAMPLES:", STEP_SAMPLES)
print("Overlap %:", 100.0 * (1.0 - STEP_SAMPLES / WINDOW_SAMPLES))
print("QC thresholds:", MIN_VALID_PCT_POST, "% valid post,", MAX_GAP_SEC_POST, "sec max gap")


WINDOW_SAMPLES: 2400
STEP_SAMPLES: 240
Overlap %: 90.0
QC thresholds: 70.0 % valid post, 120.0 sec max gap


In [18]:
def load_ctg_record(record_id: str, data_dir: Path):
    rec_path = str((data_dir / str(record_id)).resolve())
    p_signal, fields = wfdb.rdsamp(rec_path)

    sig_names = [s.upper() for s in fields.get("sig_name", [])]

    def find_idx(keys):
        for i, name in enumerate(sig_names):
            for k in keys:
                if k in name:
                    return i
        return None

    fhr_idx = find_idx(["FHR", "FETAL"])
    uc_idx  = find_idx(["UC", "TOCO", "UA", "UTERINE"])

    fs = float(fields.get("fs", FS))
    return p_signal, fields, fhr_idx, uc_idx, fs

print("Record loader ready")


Record loader ready


In [19]:
# --- Preprocess config (same as Block B) ---
FHR_MIN, FHR_MAX = 80.0, 240.0
UC_MIN, UC_MAX = 0.0, 100.0
FHR_MISSING_SENTINEL = 0.0

MAX_INTERP_GAP_SEC = 30.0
GAUSS_SIGMA_SEC = 1.5

def runs_of_true(mask: np.ndarray):
    if mask.size == 0:
        return np.array([], dtype=int), np.array([], dtype=int)
    m = mask.astype(np.int8)
    diff = np.diff(np.r_[0, m, 0])
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    return starts, ends

def interp_nan_with_gap_limit(y: np.ndarray, max_gap_samples: int):
    y = np.asarray(y, dtype=float).copy()
    n = len(y)
    isnan = np.isnan(y)
    if not isnan.any():
        return y

    starts, ends = runs_of_true(isnan)
    x = np.arange(n)
    good = ~isnan
    if good.sum() < 2:
        return y

    y_interp = np.interp(x, x[good], y[good])
    for s, e in zip(starts, ends):
        if (e - s) <= max_gap_samples:
            y[s:e] = y_interp[s:e]
    return y

def preprocess_fhr(fhr: np.ndarray, fs: float, zscore_per_record: bool = False):
    y0 = np.asarray(fhr, dtype=float)
    y = y0.copy()
    n = len(y)

    sentinel_mask = (y == FHR_MISSING_SENTINEL)
    nan_mask0 = np.isnan(y)
    missing_mask0 = nan_mask0 | sentinel_mask
    y[missing_mask0] = np.nan

    outlier_mask = np.isfinite(y) & ((y < FHR_MIN) | (y > FHR_MAX))
    y[outlier_mask] = np.nan

    pre_nan = np.isnan(y)
    starts, ends = runs_of_true(pre_nan)
    pre_max_gap_samples = int(np.max(ends - starts)) if len(starts) else 0
    pre_max_gap_sec = pre_max_gap_samples / fs

    # interpolate short gaps
    max_fill = max(1, int(round(MAX_INTERP_GAP_SEC * fs)))
    if np.isfinite(y).sum() >= 2:
        y = interp_nan_with_gap_limit(y, max_gap_samples=max_fill)

    # smooth
    sigma_samp = max(0.0, GAUSS_SIGMA_SEC * fs)
    if sigma_samp > 0:
        if np.isnan(y).any():
            idx = np.arange(n)
            good = np.isfinite(y)
            if good.sum() >= 2:
                y_fill = np.interp(idx, idx[good], y[good])
            else:
                y_fill = np.nan_to_num(y, nan=0.0)
            y_smooth = gaussian_filter1d(y_fill, sigma=sigma_samp, mode="nearest")
            y_smooth[np.isnan(y)] = np.nan
            y = y_smooth
        else:
            y = gaussian_filter1d(y, sigma=sigma_samp, mode="nearest")

    if zscore_per_record:
        mu = np.nanmean(y)
        sd = np.nanstd(y)
        if np.isfinite(sd) and sd > 0:
            y = (y - mu) / sd
        else:
            y = y * 0.0

    post_nan = np.isnan(y)
    starts2, ends2 = runs_of_true(post_nan)
    post_max_gap_samples = int(np.max(ends2 - starts2)) if len(starts2) else 0
    post_max_gap_sec = post_max_gap_samples / fs

    qc = {
        "fhr_missing_pct": float(100.0 * pre_nan.mean()) if n else np.nan,
        "fhr_max_missing_gap_sec": float(pre_max_gap_sec),
        "fhr_remaining_nan_pct": float(100.0 * post_nan.mean()) if n else np.nan,
        "fhr_valid_pct_post": float(100.0 * (~post_nan).mean()) if n else np.nan,
        "fhr_max_missing_gap_sec_post": float(post_max_gap_sec),
    }
    return y, qc

def preprocess_uc(uc: np.ndarray, fs: float, zscore_per_record: bool = False):
    y = np.asarray(uc, dtype=float).copy()
    n = len(y)

    outliers = np.isfinite(y) & ((y < UC_MIN) | (y > UC_MAX))
    y[outliers] = np.nan

    sigma_samp = max(0.0, GAUSS_SIGMA_SEC * fs)
    if sigma_samp > 0:
        if np.isnan(y).any():
            idx = np.arange(n)
            good = np.isfinite(y)
            if good.sum() >= 2:
                y_fill = np.interp(idx, idx[good], y[good])
            else:
                y_fill = np.nan_to_num(y, nan=0.0)
            y_smooth = gaussian_filter1d(y_fill, sigma=sigma_samp, mode="nearest")
            y_smooth[np.isnan(y)] = np.nan
            y = y_smooth
        else:
            y = gaussian_filter1d(y, sigma=sigma_samp, mode="nearest")

    if zscore_per_record:
        mu = np.nanmean(y)
        sd = np.nanstd(y)
        if np.isfinite(sd) and sd > 0:
            y = (y - mu) / sd
        else:
            y = y * 0.0

    qc = {
        "uc_outliers_pct": float(100.0 * outliers.mean()) if n else np.nan,
        "uc_missing_pct": float(100.0 * np.isnan(y).mean()) if n else np.nan,
    }
    return y, qc

print("Preprocessing helpers ready")


Preprocessing helpers ready


In [20]:
def window_qc_post(x: np.ndarray, fs: float):
    """
    QC computed on a *post-preprocess* signal where NaNs remain for long gaps.
    Returns:
      remaining_nan_pct, valid_pct_post, max_gap_sec_post
    """
    n = len(x)
    nan_mask = np.isnan(x)
    remaining_nan_pct = float(100.0 * nan_mask.mean()) if n else np.nan
    valid_pct_post = float(100.0 * (~nan_mask).mean()) if n else np.nan

    starts, ends = runs_of_true(nan_mask)
    max_gap_samples = int(np.max(ends - starts)) if len(starts) else 0
    max_gap_sec = float(max_gap_samples / fs) if fs else np.nan

    return remaining_nan_pct, valid_pct_post, max_gap_sec


In [21]:
def make_windows_for_record(
    record_id: str,
    fhr_clean: np.ndarray,
    uc_clean: np.ndarray,
    fs: float,
    outcome_label,
    drop_label_nan: bool = False
):
    """
    Returns list of window rows (dicts) for manifest.
    """
    rows = []
    n = len(fhr_clean)

    if drop_label_nan and (pd.isna(outcome_label) or str(outcome_label).strip() == ""):
        return rows

    # sliding windows
    w = WINDOW_SAMPLES
    step = STEP_SAMPLES

    window_idx = 0
    for start in range(0, n - w + 1, step):
        end = start + w

        fhr_w = fhr_clean[start:end]
        uc_w  = uc_clean[start:end]

        fhr_nan_pct, fhr_valid_post, fhr_max_gap_post = window_qc_post(fhr_w, fs)

        keep = (fhr_valid_post >= MIN_VALID_PCT_POST) and (fhr_max_gap_post <= MAX_GAP_SEC_POST)

        rows.append({
            "record_id": str(record_id),
            "window_idx": int(window_idx),
            "start_sample": int(start),
            "end_sample": int(end),
            "start_min": float(start / fs / 60.0),
            "end_min": float(end / fs / 60.0),
            "fs": float(fs),
            "window_minutes": float(WINDOW_MIN),
            "step_minutes": float(STEP_MIN),

            "outcome_label": outcome_label,

            # QC (post-preprocess)
            "fhr_remaining_nan_pct_window": float(fhr_nan_pct),
            "fhr_valid_pct_post_window": float(fhr_valid_post),
            "fhr_max_missing_gap_sec_post_window": float(fhr_max_gap_post),

            "keep_window": bool(keep),
        })

        window_idx += 1

    return rows

print("Windowing helper ready")


Windowing helper ready


In [22]:
# Choose whether to include unknown-label records in the manifest
DROP_UNKNOWN_LABELS = True  # recommended for supervised learning manifests

manifest_rows = []
record_failures = []

# Build a quick mapping from record_id -> label
label_map = dict(zip(dfA["record_id"].astype(str), dfA["outcome_label"]))

for rid in dfA["record_id"].astype(str).tolist():
    outcome_label = label_map.get(rid, np.nan)

    if DROP_UNKNOWN_LABELS and (pd.isna(outcome_label) or str(outcome_label).strip() == ""):
        continue

    try:
        sig, fields, fhr_idx, uc_idx, fs = load_ctg_record(rid, data_dir)
        if fhr_idx is None or uc_idx is None:
            record_failures.append({"record_id": rid, "error": "Missing FHR/UC channel"})
            continue

        fhr_raw = sig[:, fhr_idx]
        uc_raw  = sig[:, uc_idx]

        # Preprocess full record once, then window it
        # zscore_per_record=False for RF-feature pipeline; for CNN you can z-score later
        fhr_clean, _ = preprocess_fhr(fhr_raw, fs=fs, zscore_per_record=False)
        uc_clean, _  = preprocess_uc(uc_raw,  fs=fs, zscore_per_record=False)

        rows = make_windows_for_record(
            record_id=rid,
            fhr_clean=fhr_clean,
            uc_clean=uc_clean,
            fs=fs,
            outcome_label=outcome_label,
            drop_label_nan=False
        )
        manifest_rows.extend(rows)

    except Exception as e:
        record_failures.append({"record_id": rid, "error": str(e)})

manifest_df = pd.DataFrame(manifest_rows)
print("Windows created:", len(manifest_df))
print("Record failures:", len(record_failures))
manifest_df.head()


Windows created: 35787
Record failures: 0


,record_id,window_idx,start_sample,end_sample,start_min,end_min,fs,window_minutes,step_minutes,outcome_label,fhr_remaining_nan_pct_window,fhr_valid_pct_post_window,fhr_max_missing_gap_sec_post_window,keep_window
0,1001,0,0,2400,0.0,10.0,4.0,10.0,1.0,Pathological,0.0,100.0,0.0,True
1,1001,1,240,2640,1.0,11.0,4.0,10.0,1.0,Pathological,0.0,100.0,0.0,True
2,1001,2,480,2880,2.0,12.0,4.0,10.0,1.0,Pathological,0.0,100.0,0.0,True
3,1001,3,720,3120,3.0,13.0,4.0,10.0,1.0,Pathological,0.0,100.0,0.0,True
4,1001,4,960,3360,4.0,14.0,4.0,10.0,1.0,Pathological,0.0,100.0,0.0,True


In [23]:
manifest_path = out_dir / "segmentation_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print("Saved:", manifest_path)

# Summary
total_windows = len(manifest_df)
kept_windows = int(manifest_df["keep_window"].sum()) if total_windows else 0
dropped_windows = total_windows - kept_windows

print("\nWINDOW SUMMARY")
print("Total windows:", total_windows)
print("Kept windows:", kept_windows)
print("Dropped windows:", dropped_windows)

print("\nKEPT WINDOWS BY CLASS")
print(manifest_df.loc[manifest_df["keep_window"]].groupby("outcome_label")["keep_window"].count())

print("\nDROP RATE BY CLASS (%)")
drop_rate = manifest_df.groupby("outcome_label")["keep_window"].apply(lambda s: 100.0*(1.0 - s.mean()))
print(drop_rate)


Saved: /home/naem_haq/projects/CTG/outputs/segmentation_manifest.csv

WINDOW SUMMARY
Total windows: 35787
Kept windows: 29989
Dropped windows: 5798

KEPT WINDOWS BY CLASS
outcome_label
Normal          24358
Pathological     5631
Name: keep_window, dtype: int64

DROP RATE BY CLASS (%)
outcome_label
Normal          15.394234
Pathological    19.522653
Name: keep_window, dtype: float64


In [24]:
if record_failures:
    fail_df = pd.DataFrame(record_failures)
    print("Example record failures:")
    print(fail_df.head(10).to_string(index=False))

# Show worst-quality windows (highest remaining NaNs)
print("\nWORST WINDOWS (highest remaining NaNs)")
cols = [
    "record_id", "window_idx",
    "outcome_label",
    "fhr_remaining_nan_pct_window",
    "fhr_valid_pct_post_window",
    "fhr_max_missing_gap_sec_post_window",
    "keep_window"
]
print(manifest_df.sort_values("fhr_remaining_nan_pct_window", ascending=False)[cols].head(15).to_string(index=False))



WORST WINDOWS (highest remaining NaNs)
record_id  window_idx outcome_label  fhr_remaining_nan_pct_window  fhr_valid_pct_post_window  fhr_max_missing_gap_sec_post_window  keep_window
     2046          68  Pathological                         100.0                        0.0                                600.0        False
     2046          67  Pathological                         100.0                        0.0                                600.0        False
     2046          66  Pathological                         100.0                        0.0                                600.0        False
     2046          65  Pathological                         100.0                        0.0                                600.0        False
     2046          69  Pathological                         100.0                        0.0                                600.0        False
     2046          64  Pathological                         100.0                        0.0          

## Notes carried forward

- Windowing is overlapping: 10 min context, 1 min updates (real-time friendly).
- We preprocess per-record once, then window the cleaned signals.
- Per-window QC is computed on post-preprocess signals (NaNs preserved for long gaps).
- The manifest is the single source of truth for later blocks:
  - Block D: compute features for windows where keep_window=True
  - Block E: train RF using windows (split by record_id!)
  - Block F: create real-time risk timeline using window timestamps


In [25]:
rid = "1001"
sig, fields, fhr_idx, uc_idx, fs = load_ctg_record(rid, data_dir)
fhr_raw = sig[:, fhr_idx]
fhr_clean, qc_rec = preprocess_fhr(fhr_raw, fs=fs, zscore_per_record=False)

# Compare record-level remaining NaNs
print("Record-level remaining NaN %:", qc_rec["fhr_remaining_nan_pct"])

# Now inspect a few windows directly
for widx in [0, 10, 20]:
    start = widx * STEP_SAMPLES
    end = start + WINDOW_SAMPLES
    fhr_w = fhr_clean[start:end]
    print(widx, "window NaN %:", 100*np.isnan(fhr_w).mean(),
          "max gap sec:", (max(runs_of_true(np.isnan(fhr_w))[1]-runs_of_true(np.isnan(fhr_w))[0]) / fs) if np.isnan(fhr_w).any() else 0)


Record-level remaining NaN %: 14.052083333333334
0 window NaN %: 0.0 max gap sec: 0
10 window NaN %: 0.0 max gap sec: 0
20 window NaN %: 5.458333333333333 max gap sec: 32.75


In [26]:
(manifest_df["end_sample"] - manifest_df["start_sample"]).unique()
(manifest_df.groupby("record_id")["start_sample"].diff().dropna().unique())


array([240.])

In [27]:
err = (manifest_df["start_min"] - manifest_df["start_sample"]/(manifest_df["fs"]*60)).abs().max()
err


np.float64(0.0)

In [28]:
chk = (manifest_df["fhr_valid_pct_post_window"] + manifest_df["fhr_remaining_nan_pct_window"])
chk.min(), chk.max()


(np.float64(99.99999999999999), np.float64(100.00000000000001))